# State Passing, Hand-offs, and Agent Contracts

Multi-agent workflows fail in subtle ways when agents pass **too much** (full chat logs), **too little** (missing order id), or **the wrong shape** (unstructured blobs). Production systems fix this with three disciplines:

1. **Shared workflow state** — a single structured object every agent reads and writes through defined keys.
2. **Handoff packets** — distilled context passed between agents, not raw conversation history.
3. **Agent contracts** — explicit schemas for what each agent may read, write, and return.

This notebook implements a **ShopCo Support Pipeline** where four agents collaborate through validated state and contracts: intake → research → compose → validate. Everything is self-contained plain Python.

In [ ]:
"""
ShopCo Support Pipeline — state, handoffs, and agent contracts.
"""

import copy
import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


ORDERS: dict[str, dict[str, Any]] = {
    "ORD-1001": {"customer": "alice@shop.com", "status": "shipped", "amount": 1299.00},
    "ORD-1002": {"customer": "bob@shop.com", "status": "processing", "amount": 449.00},
}

POLICIES = [
    {"id": "ret-02", "text": "Returns within 30 days if unused."},
    {"id": "cancel-03", "text": "Cancellations allowed only before shipment."},
]

print(f"Backend ready — {len(ORDERS)} orders, {len(POLICIES)} policies")

---

## 1. The Problem — What Goes Wrong Without Contracts

| Failure mode | Symptom | Root cause |
|--------------|---------|------------|
| **Context dump** | 50k tokens per handoff | Passing full chat history |
| **Lost entities** | "Which order?" loops | order_id not in shared state |
| **Schema drift** | Downstream KeyError | Agent B expects `order` but got `order_data` |
| **Silent overwrite** | Research notes vanish | Two agents write same key without rules |
| **Prompt injection via state** | Tool hijack | Untrusted text copied into state unchecked |

**Fix:** treat workflow state like an API between agents — versioned keys, validated handoffs, append-only audit log.

---

## 2. Workflow State vs Agent-Local State

```
┌─────────────────────────────────────────────────────────┐
│  WorkflowState (shared, orchestrator-owned)             │
│  ├── user_query                                         │
│  ├── entities { order_id, email, intent }               │
│  ├── artifacts { order_info, policy_hits, draft }     │
│  ├── handoff_log[]                                      │
│  └── status: intake | researching | composing | done    │
└─────────────────────────────────────────────────────────┘
         ▲                    ▲                    ▲
         │ read/write         │ read/write         │ read only
    intake_agent        research_agent         validate_agent
    (local: regex        (local: tool cache)   (local: rule list)
     patterns)
```

| State type | Scope | Examples |
|------------|-------|----------|
| **Workflow state** | Entire run, all agents | `entities`, `artifacts`, `status` |
| **Agent-local state** | One agent invocation | Temp variables, tool retry count |
| **Durable state** | Across sessions | `thread_id`, checkpointed WorkflowState |

Agents should **never** keep critical facts only in agent-local memory — persist extracted entities to workflow state immediately.

In [ ]:
class WorkflowStatus(str, Enum):
    INTAKE = "intake"
    RESEARCHING = "researching"
    COMPOSING = "composing"
    VALIDATING = "validating"
    DONE = "done"
    REJECTED = "rejected"


@dataclass
class WorkflowState:
    thread_id: str
    user_query: str
    entities: dict[str, Any] = field(default_factory=dict)
    artifacts: dict[str, Any] = field(default_factory=dict)
    handoff_log: list[dict[str, Any]] = field(default_factory=list)
    status: WorkflowStatus = WorkflowStatus.INTAKE
    final_answer: str = ""
    revision_count: int = 0

    def snapshot(self) -> dict[str, Any]:
        return {
            "thread_id": self.thread_id,
            "status": self.status.value,
            "entities": self.entities,
            "artifact_keys": list(self.artifacts.keys()),
            "handoffs": len(self.handoff_log),
        }


state = WorkflowState(
    thread_id=str(uuid.uuid4())[:8],
    user_query="Where is order ORD-1001 and can I return it?",
)
print(pretty(state.snapshot()))

---

## 3. Handoff Packets — What to Pass Between Agents

A **handoff** is a structured message, not a forwarded chat thread.

| Field | Purpose |
|-------|--------|
| `from_agent` / `to_agent` | Audit trail |
| `task_summary` | One-sentence distilled subtask |
| `payload` | Structured facts downstream needs (order_id, intent) |
| `constraints` | read_only, cite_policy_ids, max_words |
| `expected_output` | Keys the receiver must write (e.g. `order_info`) |

**Do not pass:** full message history, raw tool JSON dumps, secrets, or unvalidated user strings as instructions.

In [ ]:
@dataclass
class HandoffPacket:
    handoff_id: str
    from_agent: str
    to_agent: str
    task_summary: str
    payload: dict[str, Any]
    constraints: list[str] = field(default_factory=list)
    expected_output: list[str] = field(default_factory=list)
    created_at: str = field(
        default_factory=lambda: datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    )


GOOD_HANDOFF = HandoffPacket(
    handoff_id="h-001",
    from_agent="intake_agent",
    to_agent="research_agent",
    task_summary="Look up ORD-1001 and find return policy for shipped orders",
    payload={"order_id": "ORD-1001", "intent": "order_status_and_return", "email": "alice@shop.com"},
    constraints=["read_only", "cite_policy_ids"],
    expected_output=["order_info", "policy_hits"],
)

BAD_HANDOFF = {
    "from": "intake",
    "messages": ["user said a lot of stuff..." * 50],  # context dump
    "data": "maybe order 1001?",  # unstructured
}

print("Good handoff:")
print(pretty(GOOD_HANDOFF))
print(f"\nBad handoff keys: {list(BAD_HANDOFF.keys())} — no contract, no expected_output")

---

## 4. Agent Contracts — Read/Write Permissions

Each agent declares a **contract**: which state keys it may read, which it may write, and required input/output fields. The orchestrator validates every transition.

In [ ]:
@dataclass
class AgentContract:
    agent_id: str
    reads: list[str]       # state paths: "entities", "artifacts.draft"
    writes: list[str]      # state paths this agent may mutate
    required_inputs: list[str]   # keys in handoff.payload
    required_outputs: list[str]  # keys agent must produce in artifacts
    description: str = ""


CONTRACTS: dict[str, AgentContract] = {
    "intake_agent": AgentContract(
        agent_id="intake_agent",
        reads=["user_query"],
        writes=["entities", "status"],
        required_inputs=["user_query"],
        required_outputs=["intent"],
        description="Extract entities and classify intent from raw query",
    ),
    "research_agent": AgentContract(
        agent_id="research_agent",
        reads=["entities", "user_query"],
        writes=["artifacts.order_info", "artifacts.policy_hits", "status"],
        required_inputs=["intent"],
        required_outputs=["order_info", "policy_hits"],
        description="Call tools and store structured research artifacts",
    ),
    "compose_agent": AgentContract(
        agent_id="compose_agent",
        reads=["entities", "artifacts.order_info", "artifacts.policy_hits", "user_query"],
        writes=["artifacts.draft_response", "status"],
        required_inputs=["order_info", "policy_hits"],
        required_outputs=["draft_response"],
        description="Write customer-facing draft from research artifacts",
    ),
    "validate_agent": AgentContract(
        agent_id="validate_agent",
        reads=["artifacts.draft_response", "entities", "artifacts.policy_hits"],
        writes=["final_answer", "status", "revision_count"],
        required_inputs=["draft_response"],
        required_outputs=["final_answer"],
        description="Check draft against policy citations and quality rules",
    ),
}

for agent_id, c in CONTRACTS.items():
    print(f"{agent_id:<16} reads={c.reads}  writes={c.writes}")

---

## 5. Contract Validation at Boundaries

The orchestrator validates handoffs **before** invoking the next agent and validates outputs **after** each step.

In [ ]:
class ContractViolation(Exception):
    pass


def get_nested(d: dict[str, Any], path: str) -> Any:
    parts = path.split(".")
    cur: Any = d
    for p in parts:
        if p == path.split(".")[0] and p not in ("entities", "artifacts", "user_query", "status", "final_answer", "revision_count"):
            break
        if isinstance(cur, dict) and p in cur:
            cur = cur[p]
        elif p == "user_query":
            return d.get("user_query")
        elif p == "entities":
            return d.get("entities", {})
        elif p == "artifacts":
            return d.get("artifacts", {})
        elif p == "status":
            return d.get("status")
        elif p == "final_answer":
            return d.get("final_answer")
        elif p == "revision_count":
            return d.get("revision_count")
        else:
            return None
    return cur


def state_as_dict(state: WorkflowState) -> dict[str, Any]:
    return {
        "user_query": state.user_query,
        "entities": state.entities,
        "artifacts": state.artifacts,
        "status": state.status,
        "final_answer": state.final_answer,
        "revision_count": state.revision_count,
    }


def validate_handoff(packet: HandoffPacket, contract: AgentContract) -> list[str]:
    errors: list[str] = []
    for key in contract.required_inputs:
        if key not in packet.payload and key != "user_query":
            errors.append(f"Missing required input '{key}' in handoff payload")
    if packet.to_agent != contract.agent_id:
        errors.append(f"Handoff target {packet.to_agent} != contract {contract.agent_id}")
    for out in contract.required_outputs:
        if out not in packet.expected_output:
            errors.append(f"expected_output should include '{out}'")
    return errors


def validate_agent_output(
    contract: AgentContract,
    produced: dict[str, Any],
) -> list[str]:
    errors: list[str] = []
    for key in contract.required_outputs:
        if key not in produced:
            errors.append(f"Agent did not produce required output '{key}'")
    return errors


errs = validate_handoff(GOOD_HANDOFF, CONTRACTS["research_agent"])
print("Validation errors (good handoff):", errs or "none")

bad_packet = HandoffPacket(
    handoff_id="h-bad", from_agent="intake_agent", to_agent="research_agent",
    task_summary="research", payload={}, expected_output=[],
)
print("Validation errors (bad handoff):", validate_handoff(bad_packet, CONTRACTS["research_agent"]))

---

## 6. Immutable State Updates

Prefer **copy-on-write** or explicit merge functions over mutating shared dicts in place. This makes debugging and checkpointing predictable.

In [ ]:
def merge_entities(state: WorkflowState, updates: dict[str, Any]) -> WorkflowState:
    new_state = copy.deepcopy(state)
    new_state.entities = {**state.entities, **updates}
    return new_state


def merge_artifacts(state: WorkflowState, updates: dict[str, Any]) -> WorkflowState:
    new_state = copy.deepcopy(state)
    new_state.artifacts = {**state.artifacts, **updates}
    return new_state


def log_handoff(state: WorkflowState, packet: HandoffPacket) -> WorkflowState:
    new_state = copy.deepcopy(state)
    new_state.handoff_log.append({
        "id": packet.handoff_id,
        "from": packet.from_agent,
        "to": packet.to_agent,
        "task": packet.task_summary,
        "payload_keys": list(packet.payload.keys()),
    })
    return new_state


s1 = merge_entities(state, {"order_id": "ORD-1001", "intent": "order_and_return"})
print("Before:", state.entities)
print("After: ", s1.entities)
print("Original unchanged:", state.entities)

---

## 7. L2 Tools — Research Integrations

In [ ]:
def lookup_order(order_id: str) -> dict[str, Any]:
    oid = order_id.upper()
    order = ORDERS.get(oid)
    if not order:
        return {"found": False, "error": f"Order {oid} not found"}
    return {"found": True, "order_id": oid, **order}


def search_policy(query: str) -> list[dict[str, str]]:
    terms = [t for t in re.split(r"\W+", query.lower()) if len(t) > 2]
    hits = [p for p in POLICIES if any(t in p["text"].lower() for t in terms)]
    return hits or POLICIES[:1]


print(lookup_order("ORD-1001"))
print(search_policy("return"))

---

## 8. Agent Implementations — Contract-Compliant Steps

Each agent reads from state, produces structured output, and returns an updated state. No agent reads the full handoff log — only the orchestrator does.

In [ ]:
def extract_order_id(text: str) -> str | None:
    m = re.search(r"ORD-\d{4}", text.upper())
    return m.group(0) if m else None


def extract_email(text: str) -> str | None:
    m = re.search(r"[\w.-]+@[\w.-]+\.[a-zA-Z]{2,}", text)
    return m.group(0).lower() if m else None


def classify_intent(query: str) -> str:
    q = query.lower()
    if "ord" in q and any(w in q for w in ("return", "cancel")):
        return "order_status_and_return"
    if "ord" in q or "order" in q:
        return "order_status"
    if any(w in q for w in ("return", "policy", "cancel")):
        return "policy_question"
    return "general"


def run_intake_agent(state: WorkflowState) -> tuple[WorkflowState, dict[str, Any]]:
    intent = classify_intent(state.user_query)
    entities = {
        "intent": intent,
        "order_id": extract_order_id(state.user_query),
        "email": extract_email(state.user_query),
    }
    entities = {k: v for k, v in entities.items() if v is not None}
    new_state = merge_entities(state, entities)
    new_state.status = WorkflowStatus.RESEARCHING
    output = {"intent": intent}
    return new_state, output


def run_research_agent(state: WorkflowState, handoff: HandoffPacket) -> tuple[WorkflowState, dict[str, Any]]:
    output: dict[str, Any] = {}
    order_id = handoff.payload.get("order_id") or state.entities.get("order_id")
    if order_id:
        output["order_info"] = lookup_order(order_id)
    else:
        output["order_info"] = {"found": False, "skipped": True, "reason": "no order_id in context"}
    intent = handoff.payload.get("intent", "general")
    query = "return" if "return" in intent else state.user_query
    output["policy_hits"] = search_policy(query)
    new_state = merge_artifacts(state, output)
    new_state.status = WorkflowStatus.COMPOSING
    return new_state, output


def run_compose_agent(state: WorkflowState) -> tuple[WorkflowState, dict[str, Any]]:
    order = state.artifacts.get("order_info", {})
    policies = state.artifacts.get("policy_hits", [])
    cites = ", ".join(f"[{p['id']}]" for p in policies)

    if order.get("found"):
        draft = (
            f"Your order {order['order_id']} is currently {order['status']} "
            f"(${order['amount']:.2f}). "
            f"Policy: {policies[0]['text']} {cites}."
        )
    else:
        draft = f"Per ShopCo policy {cites}: {policies[0]['text'] if policies else 'contact support'}."

    new_state = merge_artifacts(state, {"draft_response": draft})
    new_state.status = WorkflowStatus.VALIDATING
    return new_state, {"draft_response": draft}


def run_validate_agent(state: WorkflowState) -> tuple[WorkflowState, dict[str, Any]]:
    draft = state.artifacts.get("draft_response", "")
    policies = state.artifacts.get("policy_hits", [])
    policy_ids = [p["id"] for p in policies]
    errors: list[str] = []

    if len(draft) < 20:
        errors.append("draft too short")
    if policies and not any(f"[{pid}]" in draft for pid in policy_ids):
        errors.append("missing policy citation")
    if "ignore instructions" in draft.lower():
        errors.append("injection detected in draft")

    new_state = copy.deepcopy(state)
    if errors:
        new_state.status = WorkflowStatus.REJECTED
        new_state.revision_count += 1
        return new_state, {"approved": False, "errors": errors}

    new_state.final_answer = draft
    new_state.status = WorkflowStatus.DONE
    return new_state, {"final_answer": draft, "approved": True}

---

## 9. Pipeline Orchestrator — State Machine with Handoffs

The orchestrator owns transitions: validate contract → build handoff → run agent → validate output → update state.

In [ ]:
PIPELINE = ["intake_agent", "research_agent", "compose_agent", "validate_agent"]
MAX_REVISIONS = 2


@dataclass
class PipelineResult:
    state: WorkflowState
    success: bool
    trace: list[str]


class ShopCoPipelineOrchestrator:
    def __init__(self, max_revisions: int = MAX_REVISIONS):
        self.max_revisions = max_revisions

    def _build_handoff(self, from_agent: str, to_agent: str, state: WorkflowState) -> HandoffPacket:
        if to_agent == "research_agent":
            return HandoffPacket(
                handoff_id=str(uuid.uuid4())[:8],
                from_agent=from_agent,
                to_agent=to_agent,
                task_summary=f"Research for intent: {state.entities.get('intent', 'general')}",
                payload=dict(state.entities),
                constraints=["read_only", "cite_policy_ids"],
                expected_output=["order_info", "policy_hits"],
            )
        if to_agent == "compose_agent":
            return HandoffPacket(
                handoff_id=str(uuid.uuid4())[:8],
                from_agent=from_agent,
                to_agent=to_agent,
                task_summary="Compose customer response from research artifacts",
                payload={
                    "order_info": state.artifacts.get("order_info"),
                    "policy_hits": state.artifacts.get("policy_hits"),
                },
                constraints=["cite_policy_ids", "max_words:150"],
                expected_output=["draft_response"],
            )
        if to_agent == "validate_agent":
            return HandoffPacket(
                handoff_id=str(uuid.uuid4())[:8],
                from_agent=from_agent,
                to_agent=to_agent,
                task_summary="Validate draft against quality and citation rules",
                payload={"draft_response": state.artifacts.get("draft_response", "")},
                constraints=["reject_on_missing_citation"],
                expected_output=["final_answer"],
            )
        return HandoffPacket(
            handoff_id=str(uuid.uuid4())[:8],
            from_agent=from_agent,
            to_agent=to_agent,
            task_summary="Process user query",
            payload={"user_query": state.user_query},
            expected_output=["intent"],
        )

    def run(self, user_query: str, thread_id: str | None = None) -> PipelineResult:
        state = WorkflowState(thread_id=thread_id or str(uuid.uuid4())[:8], user_query=user_query)
        trace: list[str] = [f"start:{state.thread_id}"]
        prev_agent = "user"

        while state.revision_count <= self.max_revisions:
            # Intake
            contract = CONTRACTS["intake_agent"]
            state, out = run_intake_agent(state)
            errs = validate_agent_output(contract, out)
            if errs:
                return PipelineResult(state, False, trace + [f"intake failed: {errs}"])
            trace.append(f"intake → entities={state.entities}")
            prev_agent = "intake_agent"

            # Research
            handoff = self._build_handoff(prev_agent, "research_agent", state)
            state = log_handoff(state, handoff)
            errs = validate_handoff(handoff, CONTRACTS["research_agent"])
            if errs:
                return PipelineResult(state, False, trace + [f"handoff invalid: {errs}"])
            state, out = run_research_agent(state, handoff)
            errs = validate_agent_output(CONTRACTS["research_agent"], out)
            if errs:
                return PipelineResult(state, False, trace + [f"research failed: {errs}"])
            trace.append(f"research → artifacts={list(state.artifacts.keys())}")
            prev_agent = "research_agent"

            # Compose
            handoff = self._build_handoff(prev_agent, "compose_agent", state)
            state = log_handoff(state, handoff)
            state, out = run_compose_agent(state)
            errs = validate_agent_output(CONTRACTS["compose_agent"], out)
            if errs:
                return PipelineResult(state, False, trace + [f"compose failed: {errs}"])
            trace.append(f"compose → draft_len={len(out['draft_response'])}")
            prev_agent = "compose_agent"

            # Validate
            handoff = self._build_handoff(prev_agent, "validate_agent", state)
            state = log_handoff(state, handoff)
            state, out = run_validate_agent(state)

            if out.get("approved"):
                trace.append("validate → approved")
                return PipelineResult(state, True, trace)

            trace.append(f"validate → rejected: {out.get('errors')} (revision {state.revision_count})")
            if state.revision_count >= self.max_revisions:
                state.status = WorkflowStatus.REJECTED
                return PipelineResult(state, False, trace)
            state.status = WorkflowStatus.COMPOSING
            state.artifacts.pop("draft_response", None)

        return PipelineResult(state, False, trace)


orchestrator = ShopCoPipelineOrchestrator()
result = orchestrator.run("Where is order ORD-1001 and can I return it?")
print(f"Success: {result.success} | Status: {result.state.status.value}")
print(f"Answer: {result.state.final_answer}")
print(f"Handoffs: {len(result.state.handoff_log)}")

---

## 10. Inspect State Transitions

Log snapshots at each step for debugging and eval datasets.

In [ ]:
print("Trace:")
for line in result.trace:
    print(f"  {line}")

print("\nHandoff log:")
for h in result.state.handoff_log:
    print(f"  {h['from']} → {h['to']}  [{h['id']}]  keys={h['payload_keys']}")

print("\nFinal entities:", result.state.entities)
print("Final artifacts:", list(result.state.artifacts.keys()))

---

## 11. Serialization — Checkpointing State

Persist `WorkflowState` as JSON for thread resume, human review, or crash recovery.

In [ ]:
def serialize_state(state: WorkflowState) -> str:
    return json.dumps({
        "thread_id": state.thread_id,
        "user_query": state.user_query,
        "entities": state.entities,
        "artifacts": state.artifacts,
        "handoff_log": state.handoff_log,
        "status": state.status.value,
        "final_answer": state.final_answer,
        "revision_count": state.revision_count,
    }, indent=2)


def deserialize_state(data: str) -> WorkflowState:
    d = json.loads(data)
    return WorkflowState(
        thread_id=d["thread_id"],
        user_query=d["user_query"],
        entities=d.get("entities", {}),
        artifacts=d.get("artifacts", {}),
        handoff_log=d.get("handoff_log", []),
        status=WorkflowStatus(d.get("status", "intake")),
        final_answer=d.get("final_answer", ""),
        revision_count=d.get("revision_count", 0),
    )


blob = serialize_state(result.state)
restored = deserialize_state(blob)
print(f"Round-trip OK: {restored.thread_id == result.state.thread_id}")
print(f"Status: {restored.status.value} | Answer length: {len(restored.final_answer)}")

---

## 12. Contract Versioning

When agent outputs evolve, version contracts to avoid breaking downstream agents.

In [ ]:
@dataclass
class VersionedContract:
    version: str
    contract: AgentContract


RESEARCH_CONTRACT_V1 = VersionedContract(
    version="1.0",
    contract=CONTRACTS["research_agent"],
)

RESEARCH_CONTRACT_V2 = VersionedContract(
    version="2.0",
    contract=AgentContract(
        agent_id="research_agent",
        reads=["entities", "user_query"],
        writes=["artifacts.order_info", "artifacts.policy_hits", "artifacts.shipping_estimate", "status"],
        required_inputs=["intent"],
        required_outputs=["order_info", "policy_hits", "shipping_estimate"],
        description="v2 adds shipping_estimate artifact",
    ),
)

print(f"v1 outputs: {RESEARCH_CONTRACT_V1.contract.required_outputs}")
print(f"v2 outputs: {RESEARCH_CONTRACT_V2.contract.required_outputs}")
print("compose_agent must be updated when upgrading research v1 → v2")

---

## 13. Multiple Queries — State Isolation

Each run gets a fresh `WorkflowState`. Never share mutable state between concurrent threads.

In [ ]:
queries = [
    "Status of ORD-1002 please",
    "What is your return policy?",
    "Order ORD-1001 — can I cancel?",
]

print(f"{'Query':<45} {'Status':<12} {'Handoffs':<8} OK")
print("-" * 75)
for q in queries:
    r = orchestrator.run(q)
    print(f"{q[:45]:<45} {r.state.status.value:<12} {len(r.state.handoff_log):<8} {'✓' if r.success else '✗'}")

---

## 14. Rejection and Revision Loop

When `validate_agent` rejects a draft, the orchestrator clears the draft and sends work back to `compose_agent` — state carries `revision_count` so loops terminate.

In [ ]:
# Force a rejection by patching compose to omit citations
def run_compose_bad(state: WorkflowState) -> tuple[WorkflowState, dict[str, Any]]:
    draft = "Your order is shipped. No citations here."
    new_state = merge_artifacts(state, {"draft_response": draft})
    new_state.status = WorkflowStatus.VALIDATING
    return new_state, {"draft_response": draft}


class RevisionDemoOrchestrator(ShopCoPipelineOrchestrator):
    def run(self, user_query: str, thread_id: str | None = None) -> PipelineResult:
        result = super().run(user_query, thread_id)
        return result


# Simulate validate rejection inline
demo_state = WorkflowState(thread_id="rev-demo", user_query="ORD-1001")
demo_state = merge_entities(demo_state, {"intent": "order_status", "order_id": "ORD-1001"})
demo_state = merge_artifacts(demo_state, {
    "order_info": lookup_order("ORD-1001"),
    "policy_hits": search_policy("return"),
})
demo_state, _ = run_compose_bad(demo_state)
demo_state, val_out = run_validate_agent(demo_state)

print(f"Approved: {val_out.get('approved')}")
print(f"Errors: {val_out.get('errors')}")
print(f"Revision count: {demo_state.revision_count}")
print("Orchestrator would loop back to compose_agent until approved or max revisions.")

---

## 15. Design Guidelines

| Guideline | Rationale |
|-----------|-----------|
| **Structured payload only** | Handoffs carry dicts with known keys |
| **Distill, don't dump** | task_summary + payload, not 50 messages |
| **Validate at boundaries** | Catch contract violations before side effects |
| **Append handoff log** | Audit trail for every agent transition |
| **Version contracts** | Safe evolution of agent outputs |
| **Isolate per thread** | Fresh WorkflowState per run |
| **Cap revision loops** | revision_count + max_revisions |

---

## 16. Anti-Patterns

| Anti-pattern | Fix |
|--------------|-----|
| Pass full chat history in handoff | Extract entities + task_summary |
| Unnamed dict blobs in artifacts | Use fixed keys: order_info, policy_hits |
| Agents write any state key | Enforce AgentContract.writes |
| No expected_output on handoff | Receiver doesn't know what to produce |
| Shared global state between threads | New WorkflowState per thread_id |
| Skip validation on "simple" agents | Every boundary gets validated |

---

## 17. Optional — LLM Agents with Structured Output

Replace rule-based agents with LLM calls that return JSON matching the contract when `USE_LIVE_LLM = True`.

In [ ]:
def llm_extract_entities(query: str) -> dict[str, Any]:
    from openai import OpenAI

    client = OpenAI(api_key=OPENAI_API_KEY)
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Extract JSON with keys: intent, order_id (or null), email (or null). "
                    "intent one of: order_status, policy_question, order_status_and_return, general."
                ),
            },
            {"role": "user", "content": query},
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )
    return json.loads(resp.choices[0].message.content)


if USE_LIVE_LLM:
    print(llm_extract_entities("Where is ORD-1001? Email alice@shop.com"))
else:
    print("Offline mode — rule-based intake_agent used in pipeline.")

---

## 18. Check Your Understanding

1. What is the difference between **workflow state** and **agent-local state**?
2. What five fields belong in a **HandoffPacket**?
3. Why validate contracts at **boundaries**, not inside each agent?
4. What goes in `payload` vs what should stay out?
5. How does `revision_count` prevent infinite compose/validate loops?
6. Why version agent contracts when outputs change?
7. Name two handoff anti-patterns and their fixes.

---

## 19. Summary

Reliable multi-agent workflows treat inter-agent communication like an API:

- **WorkflowState** holds `entities`, `artifacts`, `status`, and an append-only `handoff_log`.
- **HandoffPackets** pass distilled `task_summary` + structured `payload`, not chat dumps.
- **AgentContracts** define read/write permissions and required inputs/outputs.
- The **orchestrator** validates every handoff and agent output at boundaries.
- **Serialization** enables checkpointing; **versioning** enables safe evolution.

The ShopCo pipeline ran intake → research → compose → validate with contract checks at each edge. That pattern scales to supervisor graphs, LangGraph state, and production HTTP services — the shapes change, but the contracts stay the same.